In [22]:
# CELL 1 — Imports, locked config, load Phase 1 output
import numpy as np
import torch
import torch.nn as nn
import math
import time
import json
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix

# ---- Locked architecture + training config (from final proposal) ----
N_QUBITS = 4
ENTANGLING_LAYERS = 2
D_MODEL = 64
D_FF = 128
N_HEADS = 4
N_TOKENS = 225
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
EPOCHS = 50
LR = 2e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
SEEDS = [42, 43, 44]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ---- Load Indian Pines preprocessed tensors ----
data = np.load("preprocessed/IndianPines.npz")
train_tokens = torch.tensor(data["train_tokens"], dtype=torch.float32)
train_labels = torch.tensor(data["train_labels"] - 1, dtype=torch.long)  # 0-indexed for CE loss
val_tokens = torch.tensor(data["val_tokens"], dtype=torch.float32)
val_labels = torch.tensor(data["val_labels"] - 1, dtype=torch.long)
test_tokens = torch.tensor(data["test_tokens"], dtype=torch.float32)
test_labels = torch.tensor(data["test_labels"] - 1, dtype=torch.long)

K_DIM = train_tokens.shape[-1]
N_CLASSES = int(train_labels.max().item()) + 1
print(f"Indian Pines: k={K_DIM}, classes={N_CLASSES}")
print(f"train={train_tokens.shape}, val={val_tokens.shape}, test={test_tokens.shape}")

Using device: cuda
Indian Pines: k=31, classes=16
train=torch.Size([1024, 225, 31]), val=torch.Size([1025, 225, 31]), test=torch.Size([8200, 225, 31])


In [23]:
# Cell 2 — permanent fix: zero dropout in the transformer encoder layer
class QuantFormer(nn.Module):
    def __init__(self, k_dim, n_classes):
        super().__init__()
        self.q_encoder = QuantumTokenEncoder(k_dim)
        self.pos_enc = SinusoidalPositionalEncoding(N_TOKENS, D_MODEL)
        self.encoder = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=D_FF,
            activation="relu", batch_first=True, norm_first=True,
            dropout=0.0,  # LOCKED: was an unintended PyTorch default (0.1), removed after Phase 2 diagnostic
        )
        self.classifier = nn.Linear(D_MODEL, n_classes)

    def forward(self, tokens):
        x = self.q_encoder(tokens)
        x = self.pos_enc(x)
        x = self.encoder(x)
        x = x.mean(dim=1)
        return self.classifier(x)

In [24]:
# Cell 3 — permanent fix: best-checkpoint selection built into train_one_seed
def train_one_seed(seed, k_dim, n_classes, train_tokens, train_labels, val_tokens, val_labels):
    torch.manual_seed(seed)
    model = QuantFormer(k_dim, n_classes).to(DEVICE)
    optimizer = torch.optim.Adam(get_param_groups(model), lr=LR)
    criterion = nn.CrossEntropyLoss()
    n_train = train_tokens.shape[0]
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_val_acc, best_state = -1, None

    for epoch in range(EPOCHS):
        model.train()
        perm = torch.randperm(n_train)
        epoch_loss = 0.0
        for i in range(0, n_train, TRAIN_BATCH_SIZE):
            idx = perm[i:i + TRAIN_BATCH_SIZE]
            xb, yb = train_tokens[idx].to(DEVICE), train_labels[idx].to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()
            epoch_loss += loss.item() * len(idx)

        model.eval()
        with torch.no_grad():
            val_out = model(val_tokens.to(DEVICE))
            val_loss = criterion(val_out, val_labels.to(DEVICE)).item()
            val_acc = accuracy_score(val_labels, val_out.argmax(dim=1).cpu())
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        history["train_loss"].append(epoch_loss / n_train)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  seed={seed} epoch={epoch+1}/{EPOCHS} val_acc={val_acc:.4f} (best={best_val_acc:.4f})")

    model.load_state_dict(best_state)  # LOCKED: reload best-val checkpoint before returning
    return model, history

In [25]:
# CELL 4 — Evaluation against test set

def evaluate(model, test_tokens, test_labels, n_classes):
    model.eval()
    with torch.no_grad():
        out = model(test_tokens.to(DEVICE))
        preds = out.argmax(dim=1).cpu().numpy()
    labels_np = test_labels.numpy()

    oa = accuracy_score(labels_np, preds)
    cm = confusion_matrix(labels_np, preds, labels=list(range(n_classes)))
    per_class_acc = cm.diagonal() / cm.sum(axis=1).clip(min=1)
    aa = per_class_acc.mean()
    kappa = cohen_kappa_score(labels_np, preds)

    return {"OA": oa, "AA": aa, "kappa": kappa, "confusion_matrix": cm.tolist(),
            "per_class_accuracy": per_class_acc.tolist()}

print("Cell 4 loaded: evaluate() ready.")

Cell 4 loaded: evaluate() ready.


In [26]:
# CELL 5 — Indian Pines reproduction, 3 seeds, checkpoint against paper's reported numbers
PAPER_OA, PAPER_AA, PAPER_KAPPA = 0.9908, 0.9886, 0.990

results = []
for seed in SEEDS:
    print(f"\n=== Indian Pines, seed={seed} ===")
    start = time.time()
    model, history = train_one_seed(seed, K_DIM, N_CLASSES,
                                     train_tokens, train_labels, val_tokens, val_labels)
    elapsed = time.time() - start
    metrics = evaluate(model, test_tokens, test_labels, N_CLASSES)
    metrics["seed"] = seed
    metrics["train_time_sec"] = elapsed
    results.append(metrics)
    print(f"  OA={metrics['OA']:.4f}  AA={metrics['AA']:.4f}  kappa={metrics['kappa']:.4f}  "
          f"(train time: {elapsed/60:.1f} min)")

oa_vals = [r["OA"] for r in results]
aa_vals = [r["AA"] for r in results]
kappa_vals = [r["kappa"] for r in results]

print(f"\n{'='*60}\nINDIAN PINES — 3-SEED SUMMARY (mean ± std)\n{'='*60}")
print(f"OA:    {np.mean(oa_vals):.4f} ± {np.std(oa_vals):.4f}   (paper: {PAPER_OA})")
print(f"AA:    {np.mean(aa_vals):.4f} ± {np.std(aa_vals):.4f}   (paper: {PAPER_AA})")
print(f"kappa: {np.mean(kappa_vals):.4f} ± {np.std(kappa_vals):.4f}   (paper: {PAPER_KAPPA})")

within_1pct = abs(np.mean(oa_vals) - PAPER_OA) <= 0.01
print(f"\nWithin 1% OA of paper: {'YES' if within_1pct else 'NO'} "
      f"(diff = {abs(np.mean(oa_vals) - PAPER_OA)*100:.2f} percentage points)")

with open("phase2_indian_pines_results.json", "w") as f:
    json.dump({"per_seed": results, "summary": {
        "OA_mean": float(np.mean(oa_vals)), "OA_std": float(np.std(oa_vals)),
        "AA_mean": float(np.mean(aa_vals)), "AA_std": float(np.std(aa_vals)),
        "kappa_mean": float(np.mean(kappa_vals)), "kappa_std": float(np.std(kappa_vals)),
        "within_1pct_OA": bool(within_1pct),
    }}, f, indent=2)
print("\nSaved phase2_indian_pines_results.json")


=== Indian Pines, seed=42 ===
  seed=42 epoch=1/50 val_acc=0.3824 (best=0.3824)
  seed=42 epoch=10/50 val_acc=0.7141 (best=0.7141)
  seed=42 epoch=20/50 val_acc=0.8107 (best=0.8254)
  seed=42 epoch=30/50 val_acc=0.8498 (best=0.8507)
  seed=42 epoch=40/50 val_acc=0.8507 (best=0.8741)
  seed=42 epoch=50/50 val_acc=0.8390 (best=0.8761)
  OA=0.8838  AA=0.7851  kappa=0.8673  (train time: 1.7 min)

=== Indian Pines, seed=43 ===
  seed=43 epoch=1/50 val_acc=0.4137 (best=0.4137)
  seed=43 epoch=10/50 val_acc=0.6341 (best=0.6605)
  seed=43 epoch=20/50 val_acc=0.7932 (best=0.8098)
  seed=43 epoch=30/50 val_acc=0.8722 (best=0.8839)
  seed=43 epoch=40/50 val_acc=0.9141 (best=0.9141)
  seed=43 epoch=50/50 val_acc=0.8751 (best=0.9180)
  OA=0.9028  AA=0.8140  kappa=0.8892  (train time: 1.7 min)

=== Indian Pines, seed=44 ===
  seed=44 epoch=1/50 val_acc=0.3532 (best=0.3532)
  seed=44 epoch=10/50 val_acc=0.6888 (best=0.6888)
  seed=44 epoch=20/50 val_acc=0.8585 (best=0.8585)
  seed=44 epoch=30/50 val

In [9]:
# CELL 6 — Extended-epoch diagnostic probe (seed=42 only, NOT part of locked protocol)
# Purpose: does val_acc plateau above 88% given more room, or is undertraining not the (main) story?

def train_one_seed_extended(seed, k_dim, n_classes, train_tokens, train_labels,
                             val_tokens, val_labels, n_epochs):
    torch.manual_seed(seed)
    model = QuantFormer(k_dim, n_classes).to(DEVICE)
    optimizer = torch.optim.Adam(get_param_groups(model), lr=LR)
    criterion = nn.CrossEntropyLoss()
    n_train = train_tokens.shape[0]
    history = {"epoch": [], "train_loss": [], "val_loss": [], "val_acc": []}

    for epoch in range(n_epochs):
        model.train()
        perm = torch.randperm(n_train)
        epoch_loss = 0.0
        for i in range(0, n_train, TRAIN_BATCH_SIZE):
            idx = perm[i:i + TRAIN_BATCH_SIZE]
            xb = train_tokens[idx].to(DEVICE)
            yb = train_labels[idx].to(DEVICE)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()
            epoch_loss += loss.item() * len(idx)

        model.eval()
        with torch.no_grad():
            val_out = model(val_tokens.to(DEVICE))
            val_loss = criterion(val_out, val_labels.to(DEVICE)).item()
            val_acc = accuracy_score(val_labels, val_out.argmax(dim=1).cpu())

        history["epoch"].append(epoch + 1)
        history["train_loss"].append(epoch_loss / n_train)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        # print every epoch from 45 onward so we can see the plateau (or lack of it) clearly
        if epoch + 1 >= 45 or (epoch + 1) % 10 == 0:
            print(f"epoch={epoch+1}/{n_epochs} train_loss={epoch_loss/n_train:.4f} "
                  f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    return model, history

print("Running extended probe: seed=42, 100 epochs...")
_, ext_history = train_one_seed_extended(
    42, K_DIM, N_CLASSES, train_tokens, train_labels, val_tokens, val_labels, n_epochs=100
)

# quick read: is epoch 90-100 val_acc meaningfully above epoch 50's 0.882?
print(f"\nval_acc @ epoch 50: {ext_history['val_acc'][49]:.4f}")
print(f"val_acc @ epoch 100: {ext_history['val_acc'][99]:.4f}")
print(f"gain from 50->100 epochs: {ext_history['val_acc'][99] - ext_history['val_acc'][49]:+.4f}")

Running extended probe: seed=42, 100 epochs...
epoch=10/100 train_loss=0.6448 val_loss=0.7787 val_acc=0.7307
epoch=20/100 train_loss=0.3552 val_loss=0.6111 val_acc=0.8166
epoch=30/100 train_loss=0.2430 val_loss=0.6244 val_acc=0.8224
epoch=40/100 train_loss=0.1478 val_loss=0.5283 val_acc=0.8498
epoch=45/100 train_loss=0.1215 val_loss=0.4956 val_acc=0.8673
epoch=46/100 train_loss=0.0943 val_loss=0.4035 val_acc=0.8849
epoch=47/100 train_loss=0.0707 val_loss=0.5389 val_acc=0.8527
epoch=48/100 train_loss=0.1077 val_loss=0.5421 val_acc=0.8732
epoch=49/100 train_loss=0.0850 val_loss=0.4428 val_acc=0.8937
epoch=50/100 train_loss=0.0811 val_loss=0.4752 val_acc=0.8820
epoch=51/100 train_loss=0.0738 val_loss=0.5344 val_acc=0.8849
epoch=52/100 train_loss=0.0734 val_loss=0.4776 val_acc=0.8839
epoch=53/100 train_loss=0.0997 val_loss=0.4810 val_acc=0.8790
epoch=54/100 train_loss=0.0481 val_loss=0.4744 val_acc=0.8868
epoch=55/100 train_loss=0.0560 val_loss=0.5547 val_acc=0.8732
epoch=56/100 train_loss

In [ ]:
# Cell 7 Correlate per-class accuracy (seed 42) against training sample count
with open("phase2_indian_pines_results.json") as f:
    saved = json.load(f)
seed42_result = [r for r in saved["per_seed"] if r["seed"] == 42][0]
per_class_acc = seed42_result["per_class_accuracy"]

unique, counts = np.unique(train_labels.numpy(), return_counts=True)
print(f"{'class':>6} {'train_n':>8} {'test_acc':>10}")
for cls, n_train_cls in zip(unique, counts):
    print(f"{cls:>6} {n_train_cls:>8} {per_class_acc[cls]:>10.4f}")

 class  train_n   test_acc
     0        5     0.8378
     1      143     0.8494
     2       83     0.8057
     3       24     0.8677
     4       48     0.7959
     5       73     0.9247
     6        3     0.5909
     7       48     0.9817
     8        2     0.3750
     9       97     0.8843
    10      245     0.9150
    11       59     0.7516
    12       20     0.8415
    13      126     0.9674
    14       39     0.8442
    15        9     0.7600


In [ ]:
# Cell 8 Complete the w/o-QNN diagnostic (seed=42 only)
class ClassicalMirrorEncoder(nn.Module):
    def __init__(self, k_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(k_dim, N_QUBITS), nn.Tanh(), nn.Linear(N_QUBITS, D_MODEL))
    def forward(self, tokens):
        return self.net(tokens)

torch.manual_seed(42)
model_wo_qnn = QuantFormer(K_DIM, N_CLASSES)
model_wo_qnn.q_encoder = ClassicalMirrorEncoder(K_DIM)
model_wo_qnn = model_wo_qnn.to(DEVICE)

optimizer = torch.optim.Adam(get_param_groups(model_wo_qnn), lr=LR)
criterion = nn.CrossEntropyLoss()
n_train = train_tokens.shape[0]

for epoch in range(EPOCHS):
    model_wo_qnn.train()
    perm = torch.randperm(n_train)
    for i in range(0, n_train, TRAIN_BATCH_SIZE):
        idx = perm[i:i + TRAIN_BATCH_SIZE]
        xb, yb = train_tokens[idx].to(DEVICE), train_labels[idx].to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model_wo_qnn(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_wo_qnn.parameters(), GRAD_CLIP_NORM)
        optimizer.step()

wo_qnn_metrics = evaluate(model_wo_qnn, test_tokens, test_labels, N_CLASSES)
print(f"w/o QNN — OA={wo_qnn_metrics['OA']:.4f} AA={wo_qnn_metrics['AA']:.4f} kappa={wo_qnn_metrics['kappa']:.4f}")
print(f"Full model (seed 42) was — OA=0.8804")

w/o QNN — OA=0.9150 AA=0.8395 kappa=0.9032
Full model (seed 42) was — OA=0.8804


In [ ]:
# Cell 9
def train_one_seed_with_checkpointing(seed, k_dim, n_classes, train_tokens, train_labels,
                                       val_tokens, val_labels):
    torch.manual_seed(seed)
    model = QuantFormer(k_dim, n_classes).to(DEVICE)
    optimizer = torch.optim.Adam(get_param_groups(model), lr=LR)
    criterion = nn.CrossEntropyLoss()
    n_train = train_tokens.shape[0]

    best_val_acc = -1
    best_state = None

    for epoch in range(EPOCHS):
        model.train()
        perm = torch.randperm(n_train)
        for i in range(0, n_train, TRAIN_BATCH_SIZE):
            idx = perm[i:i + TRAIN_BATCH_SIZE]
            xb, yb = train_tokens[idx].to(DEVICE), train_labels[idx].to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_out = model(val_tokens.to(DEVICE))
            val_acc = accuracy_score(val_labels, val_out.argmax(dim=1).cpu())
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 10 == 0:
            print(f"  epoch={epoch+1} val_acc={val_acc:.4f} (best so far: {best_val_acc:.4f})")

    model.load_state_dict(best_state)
    print(f"  Reloaded best checkpoint (val_acc={best_val_acc:.4f})")
    return model

model_best_ckpt = train_one_seed_with_checkpointing(
    42, K_DIM, N_CLASSES, train_tokens, train_labels, val_tokens, val_labels
)
best_ckpt_metrics = evaluate(model_best_ckpt, test_tokens, test_labels, N_CLASSES)
print(f"\nBest-checkpoint OA={best_ckpt_metrics['OA']:.4f} "
      f"(final-epoch OA was 0.8804)")

  epoch=10 val_acc=0.7307 (best so far: 0.7307)
  epoch=20 val_acc=0.8166 (best so far: 0.8283)
  epoch=30 val_acc=0.8224 (best so far: 0.8517)
  epoch=40 val_acc=0.8498 (best so far: 0.8722)
  epoch=50 val_acc=0.8820 (best so far: 0.8937)
  Reloaded best checkpoint (val_acc=0.8937)

Best-checkpoint OA=0.8859 (final-epoch OA was 0.8804)


In [ ]:
# CELL 10— Full sanity-check dump before further hypothesizing
print("="*60)
print("SANITY CHECKS — Indian Pines Phase 2 pipeline")
print("="*60)

# 1. Data shapes
print(f"\n1. Shapes:")
print(f"   train_tokens: {train_tokens.shape}  (expect: (1024, 225, k))")
print(f"   val_tokens:   {val_tokens.shape}")
print(f"   test_tokens:  {test_tokens.shape}")
print(f"   K_DIM={K_DIM}, N_CLASSES={N_CLASSES}")

# 2. Label sanity
print(f"\n2. Labels:")
print(f"   train_labels range: [{train_labels.min().item()}, {train_labels.max().item()}] "
      f"(expect 0 to {N_CLASSES-1})")
print(f"   unique train labels: {sorted(train_labels.unique().tolist())}")
assert train_labels.min() >= 0 and train_labels.max() == N_CLASSES - 1, \
    "Label range looks wrong — check the -1 offset applied in Cell 1"

# 3. Input value range (post-PCA) — should be roughly PCA-scale, not raw pixel values
print(f"\n3. Token value range (post-PCA):")
print(f"   min={train_tokens.min().item():.4f}, max={train_tokens.max().item():.4f}, "
      f"mean={train_tokens.mean().item():.4f}, std={train_tokens.std().item():.4f}")

# 4. Model parameter count — compare to paper's ~35k claim
model_check = QuantFormer(K_DIM, N_CLASSES)
total_params = sum(p.numel() for p in model_check.parameters())
quantum_params = sum(p.numel() for n, p in model_check.named_parameters() if "q_layer" in n)
classical_params = total_params - quantum_params
print(f"\n4. Parameter count:")
print(f"   total: {total_params}  (paper reports ~35k)")
print(f"   quantum circuit weights: {quantum_params}")
print(f"   classical weights: {classical_params}")

# 5. Class balance in train set
print(f"\n5. Train class distribution:")
unique, counts = np.unique(train_labels.numpy(), return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"   class {cls}: {cnt}")

# 6. Explicit list of things NOT verified against the paper (open questions, not assumptions)
print(f"\n6. UNVERIFIED against paper — these could each explain part of the gap:")
print(f"""
   [ ] Learning rate schedule — paper may use warmup/decay, we use flat LR=2e-3
   [ ] Dropout — paper's transformer encoder layer: unknown if it uses dropout
       (nn.TransformerEncoderLayer defaults to dropout=0.1 unless set to 0 explicitly —
        CHECK: did we override this? See below.)
   [ ] Weight initialization — using PyTorch defaults, paper's scheme unstated
   [ ] Batch shuffling / data augmentation during training — none applied on IP (correct
       per protocol, IP is a fidelity checkpoint) but worth confirming paper didn't do any
   [ ] Angle embedding scaling — theta = pi * tanh(...) is OUR interpretation of the paper's
       stated formula; verify this exactly matches the paper's Eq. for the token encoder
""")

# 7. Check actual dropout value in the transformer layer right now
print(f"7. Current TransformerEncoderLayer dropout: {model_check.encoder.dropout.p}")

SANITY CHECKS — Indian Pines Phase 2 pipeline

1. Shapes:
   train_tokens: torch.Size([1024, 225, 31])  (expect: (1024, 225, k))
   val_tokens:   torch.Size([1025, 225, 31])
   test_tokens:  torch.Size([8200, 225, 31])
   K_DIM=31, N_CLASSES=16

2. Labels:
   train_labels range: [0, 15] (expect 0 to 15)
   unique train labels: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

3. Token value range (post-PCA):
   min=-10.2762, max=6.6115, mean=0.0190, std=0.5719

4. Parameter count:
   total: 34984  (paper reports ~35k)
   quantum circuit weights: 24
   classical weights: 34960

5. Train class distribution:
   class 0: 5
   class 1: 143
   class 2: 83
   class 3: 24
   class 4: 48
   class 5: 73
   class 6: 3
   class 7: 48
   class 8: 2
   class 9: 97
   class 10: 245
   class 11: 59
   class 12: 20
   class 13: 126
   class 14: 39
   class 15: 9

6. UNVERIFIED against paper — these could each explain part of the gap:

   [ ] Learning rate schedule — paper may use warmup/decay, we

In [ ]:
# Cell 11- Set dropout to 0 and rerun seed 42 only
class QuantFormerNoDropout(QuantFormer):
    def __init__(self, k_dim, n_classes):
        super().__init__(k_dim, n_classes)
        self.encoder = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=D_FF,
            activation="relu", batch_first=True, norm_first=True,
            dropout=0.0,   # explicitly zeroed
        )

torch.manual_seed(42)
model_no_dropout = QuantFormerNoDropout(K_DIM, N_CLASSES).to(DEVICE)
optimizer = torch.optim.Adam(get_param_groups(model_no_dropout), lr=LR)
criterion = nn.CrossEntropyLoss()
n_train = train_tokens.shape[0]

for epoch in range(EPOCHS):
    model_no_dropout.train()
    perm = torch.randperm(n_train)
    for i in range(0, n_train, TRAIN_BATCH_SIZE):
        idx = perm[i:i + TRAIN_BATCH_SIZE]
        xb, yb = train_tokens[idx].to(DEVICE), train_labels[idx].to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model_no_dropout(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_no_dropout.parameters(), GRAD_CLIP_NORM)
        optimizer.step()

metrics_no_dropout = evaluate(model_no_dropout, test_tokens, test_labels, N_CLASSES)
print(f"No dropout — OA={metrics_no_dropout['OA']:.4f}  AA={metrics_no_dropout['AA']:.4f}  "
      f"kappa={metrics_no_dropout['kappa']:.4f}")
print(f"With dropout=0.1 (original seed 42) was — OA=0.8804")

No dropout — OA=0.9020  AA=0.8450  kappa=0.8877
With dropout=0.1 (original seed 42) was — OA=0.8804


In [ ]:
# Cell 12 - Combined: no dropout + best-checkpoint selection, seed 42
torch.manual_seed(42)
model_combined = QuantFormerNoDropout(K_DIM, N_CLASSES).to(DEVICE)
optimizer = torch.optim.Adam(get_param_groups(model_combined), lr=LR)
criterion = nn.CrossEntropyLoss()
n_train = train_tokens.shape[0]
best_val_acc, best_state = -1, None

for epoch in range(EPOCHS):
    model_combined.train()
    perm = torch.randperm(n_train)
    for i in range(0, n_train, TRAIN_BATCH_SIZE):
        idx = perm[i:i + TRAIN_BATCH_SIZE]
        xb, yb = train_tokens[idx].to(DEVICE), train_labels[idx].to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model_combined(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_combined.parameters(), GRAD_CLIP_NORM)
        optimizer.step()
    model_combined.eval()
    with torch.no_grad():
        val_acc = accuracy_score(val_labels, model_combined(val_tokens.to(DEVICE)).argmax(dim=1).cpu())
    if val_acc > best_val_acc:
        best_val_acc, best_state = val_acc, {k: v.clone() for k, v in model_combined.state_dict().items()}

model_combined.load_state_dict(best_state)
combined_metrics = evaluate(model_combined, test_tokens, test_labels, N_CLASSES)
print(f"Combined (no dropout + best checkpoint) — OA={combined_metrics['OA']:.4f}  "
      f"AA={combined_metrics['AA']:.4f}  kappa={combined_metrics['kappa']:.4f}")
print(f"Baseline was 0.8804, dropout-fix alone was 0.9020")

Combined (no dropout + best checkpoint) — OA=0.9060  AA=0.8596  kappa=0.8927
Baseline was 0.8804, dropout-fix alone was 0.9020
